In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('my_aca.csv')

In [3]:
mask = (df['Max Planet Sep'] != -99) & (df['Max Brown Dwarf Sep'] != -99)                                                     
removed_df = df[~mask]
filtered_df = df[mask]

In [4]:
output_file = 'my_new_aca.csv'
filtered_df.to_csv(output_file, index=False)

In [5]:
print(f"Original #: {len(df)}")
print(f"# after filtering: {len(filtered_df)}")
print(f"Rows removed: {len(df) - len(filtered_df)}")

print("Rows removed due to missing separation data (-99):")
print(removed_df)

Original #: 114
# after filtering: 111
Rows removed: 3
Rows removed due to missing separation data (-99):
      Star Name  Accel Significance (sigma)  Max Planet Sep  \
4     HIP 36985                       -99.0           -99.0   
73    HIP 55868                       -99.0           -99.0   
113  HIP 115004                       -99.0           -99.0   

     Max Brown Dwarf Sep  Astrometric Prediction  \
4             136.052963                     NaN   
73            199.453521                     NaN   
113           111.715584                     NaN   

                                            Kyle Notes  Bp-Rp Color  \
4    Young! 0.3 Gyr in Passeger+2019. Triple M dwar...     2.032220   
73   Single star, ages of several Gyr in Vizier, po...     0.825190   
113  On the fence with this target, has low Li EW (...     1.433483   

     Effective Temperature  Good for Gyro Ages  In TESS  TESS Sectors  \
4               3712.96500                 NaN      NaN           NaN   
7

In [6]:
from astroquery.simbad import Simbad
import time

In [7]:
def filter_single_stars(input_csv, output_csv):

    print(f"Loading {input_csv}...")
    df = pd.read_csv(input_csv)
    star_names = df['Star Name'].astype(str).unique().tolist()
    
    Simbad.reset_votable_fields()
    
    Simbad.add_votable_fields('otype')
    
    companion_map = {} 
    batch_size = 50
    
    print(f"Querying SIMBAD for {len(star_names)} stars...")
    for i in range(0, len(star_names), batch_size):
        batch = star_names[i:i + batch_size]
        try:
            result = Simbad.query_objects(batch)
            if result is None:
                continue
            
            cols = [c.lower() for c in result.colnames]

            ot_col = result.colnames[cols.index('otype')] if 'otype' in cols else \
                     result.colnames[cols.index('main_type')] if 'main_type' in cols else None
            
            id_col = result.colnames[cols.index('user_specified_id')] if 'user_specified_id' in cols else \
                     result.colnames[cols.index('main_id')] if 'main_id' in cols else None

            if ot_col is None:
                print("Error: Could not find 'otype' column in SIMBAD response.")
                continue

            for row in result:
         
                raw_id = row[id_col]
                otype = row[ot_col]
                if hasattr(raw_id, 'decode'): raw_id = raw_id.decode()
                if hasattr(otype, 'decode'): otype = otype.decode()
                

                companion_codes = ['**', 'SB*', 'EB*', 'XB*', 'CV*', 'Sy*', 'Al*', 'bL*', 'WUM*', 'EL*', 'Mul', 'binary', 'multiple']
                
                has_companion = any(code.lower() in otype.lower() for code in companion_codes)
                companion_map[raw_id] = has_companion
                
        except Exception as e:
            print(f"Error in batch starting with {batch[0]}: {e}")

    df['Is_Companion'] = df['Star Name'].astype(str).apply(lambda x: companion_map.get(x, False))

    removed_companions = df[df['Is_Companion'] == True]
    if not removed_companions.empty:
        print("\nStars identified as companions/multiples by SIMBAD and removed:")
        print(removed_companions[['Star Name']]) 
    else:
        print("\nNo stars were identified as companions.")
    
    filtered_df = df[df['Is_Companion'] == False].drop(columns=['Is_Companion'])
    
    filtered_df.to_csv(output_csv, index=False)
    print(f"Original #: {len(df)}")
    print(f"# after filtering: {len(filtered_df)} stars")
    print(f"Results saved to: {output_csv}")

if __name__ == "__main__":
    filter_single_stars('my_new_aca.csv', 'acasinglestars.csv')

Loading my_new_aca.csv...
Querying SIMBAD for 111 stars...

Stars identified as companions/multiples by SIMBAD and removed:
      Star Name
9    HIP 110716
16   HIP 112648
100   HIP 58994
Original #: 111
# after filtering: 108 stars
Results saved to: acasinglestars.csv


In [8]:
input_file = 'acasinglestars.csv'
output_file = 'finalstars.csv'
df = pd.read_csv(input_file)

min_temp = 3900
max_temp = 6000

filtered_df = df[(df['Effective Temperature'] >= min_temp) & 
                 (df['Effective Temperature'] <= max_temp)].copy()

removed_df = df[(df['Effective Temperature'] < min_temp) | 
                (df['Effective Temperature'] > max_temp) | 
                (df['Effective Temperature'].isna())]

removed_stars_list = removed_df['Star Name'].tolist()

filtered_df.to_csv(output_file, index=False)

print(f"# after filtering: {len(filtered_df)}")
print(f"The filtered data has been saved to: {output_file}")
print("\nStars removed:")
for star in removed_stars_list:
    print(f"- {star}")

# after filtering: 52
The filtered data has been saved to: finalstars.csv

Stars removed:
- HIP 47741
- HIP 84277
- HIP 80008
- HIP 19205
- HIP 65500
- HIP 63076
- HIP 72287
- HIP 55363
- HIP 59953
- HIP 110106
- HIP 25486
- HIP 13394
- HIP 55761
- HIP 45493
- HIP 6776
- HIP 69963
- HIP 79720
- HIP 83204
- HIP 25973
- HIP 101948
- HIP 2843
- HIP 71899
- HIP 22361
- HIP 41209
- HIP 18217
- HIP 73128
- HIP 23117
- HIP 5016
- HIP 63734
- HIP 21637
- HIP 70952
- HIP 6643
- HIP 112475
- HIP 38296
- HIP 18170
- HIP 103460
- HIP 101589
- HIP 51914
- HIP 6527
- HIP 5440
- HIP 73765
- HIP 10099
- HIP 21238
- HIP 16692
- HIP 25749
- HIP 89865
- HIP 75678
- HIP 27431
- HIP 27713
- HIP 25371
- HIP 26062
- HIP 23088
- HIP 45511
- HIP 87341
- HIP 23980
- HIP 45602
